# 01 — Environment Setup & Safe Execution Sandbox
**Goal**: Set up environment dependencies, verify GPU / hardware, build the sandboxed execution environment (handling time limits, memory limits, and exception tracebacks), and load programming datasets (APPS, HumanEval, MBPP).

---

## Step 1: Environment & Library Verification

In [ ]:
import sys, os

# Universal Path Resolution for Local / Kaggle Input / Kaggle Working / Colab
def setup_project_path():
    curr = os.path.abspath(os.getcwd())
    while curr != os.path.dirname(curr):
        if os.path.exists(os.path.join(curr, 'src')):
            return curr
        curr = os.path.dirname(curr)
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'src' in dirs or os.path.exists(os.path.join(root, 'models', 'loader.py')):
                return root if 'src' in dirs else os.path.dirname(root)
    for fallback in ['/kaggle/working/self-correction-llm-rl', '/kaggle/working', '/content/self-correction-llm-rl', '/content']:
        if os.path.exists(os.path.join(fallback, 'src')):
            return fallback
    return os.path.abspath('..')

repo_root = setup_project_path()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project path added: {repo_root}")

import torch
import transformers
import datasets
import peft
import trl

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Count: {torch.cuda.device_count()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
print(f"Transformers Version: {transformers.__version__}")
print(f"Datasets Version: {datasets.__version__}")
print(f"PEFT Version: {peft.__version__}")
print(f"TRL Version: {trl.__version__}")
print("All core libraries loaded successfully!")

## Step 2: Python Execution Sandbox (Verification Checks V3 & V4)
Runs untrusted Python code in isolated subprocesses with strict timeout (8s) and memory caps (1GB). Classifies execution status into AC, WA, TLE, MLE, CE, RE.

In [ ]:
from src.execution.executor import run_code, PythonSandbox
from src.execution.status import STATUSES, STATUS_DESCRIPTIONS

print("Available Execution Statuses:")
for s, desc in STATUS_DESCRIPTIONS.items():
    print(f" - {s}: {desc}")

# Test 1: All Correct (AC)
res_ac = run_code("print(2 + 2)")
print(f"\nTest 1 (AC): {res_ac}")
assert res_ac['status'] == 'AC'

# Test 2: Syntax Error (CE)
res_ce = run_code("def foo():")
print(f"Test 2 (CE): {res_ce}")
assert res_ce['status'] == 'CE'

# Test 3: Runtime Exception (RE)
res_re = run_code("print(1 / 0)")
print(f"Test 3 (RE): {res_re}")
assert res_re['status'] == 'RE'

# Test 4: Time Limit Exceeded (TLE)
res_tle = run_code("import time; time.sleep(10)", timeout=2)
print(f"Test 4 (TLE): {res_tle}")
assert res_tle['status'] == 'TLE'

print("\n--- PythonSandbox Verification ---")
sandbox = PythonSandbox(default_timeout=2.0)
sample_codes = {
    "AC": ("def add(a, b):\n    return a + b", [{"fn_name": "add", "input": [2, 3], "expected": 5}]),
    "WA": ("def add(a, b):\n    return a - b", [{"fn_name": "add", "input": [2, 3], "expected": 5}]),
    "CE": ("def add(a, b)\n    return a + b", [{"fn_name": "add", "input": [2, 3], "expected": 5}]),
    "RE": ("def add(a, b):\n    return a / 0", [{"fn_name": "add", "input": [2, 3], "expected": 5}]),
    "TLE": ("def add(a, b):\n    while True: pass", [{"fn_name": "add", "input": [2, 3], "expected": 5}]),
}
for label, (code, tests) in sample_codes.items():
    res = sandbox.run_tests(code, tests)
    print(f"[{label}] -> Status: {res.status} | Passed: {res.passed_tests}/{res.total_tests}")

print("\nVerification Checks V3 & V4 Passed!")

## Step 3: Dataset Loading & Preparation
Loads training dataset (**APPS**) and out-of-distribution evaluation benchmarks (**HumanEval**, **MBPP**).

In [ ]:
from datasets import load_dataset

print("Loading APPS training split...")
apps = load_dataset('codeparrot/apps', split='train[:2000]', trust_remote_code=True)

print("Loading HumanEval evaluation benchmark...")
humaneval = load_dataset('openai_humaneval', split='test')

print("Loading MBPP evaluation benchmark...")
mbpp = load_dataset('mbpp', split='test')

apps_clean = apps.filter(lambda x: len(x['solutions']) > 0)

print(f"\nAPPS Total Downloaded: {len(apps)}")
print(f"APPS Cleaned (with solutions): {len(apps_clean)}")
print(f"HumanEval Test Examples: {len(humaneval)}")
print(f"MBPP Test Examples: {len(mbpp)}")